# 🏆 Gemma Developer Agent: Rigorous A/B Testing Suite on Real LocBench Tasks

> ### 📌 **Notebook Version: v7.0-rigorous-ab (Updated: 2026-11-06)**
> - **真のマルチターン E2E 検証**: 見せかけの文字列チェックを完全排除し、実リポジトリ上で複数ターンのツールループを自律実行。
> - **実パッチ pytest 合否判定**: エージェントが生成した差分に対し、タスク固有のテストスイート（`pytest tests/...`）を実際に実行して合否を判定。
> - **3 大バリアントの科学的 A/B 比較**:
>   1. **Variant 0 (Baseline / V9.2)**: 現行本番プロンプト (思考制約なし・One-Shot なし)
>   2. **Variant 1 (B 案 / 最小差分安全版)**: 現行 + One-Shot JSON 例 + 配列引数禁止ガード
>   3. **Variant 2 (C 案検証 / 2文制限版)**: Variant 1 + 「思考は最大 2 文まで (Concise Reasoning)」
> - **評価メトリクス**: 解決率 (Pass Rate %)、パッチ生成率 (%)、ツール構文エラー率 (%)、平均ステップ秒数、平均ターン数。

In [ ]:
# 1. 環境初期化 & 依存パッケージ導入 & Kaggle API ステータスチェック
import os
import sys
import time
import json
import re
import gc
import subprocess
from pathlib import Path

print("[Setup] 依存パッケージをインストール中 (transformers, bitsandbytes, pytest, kaggle, etc.)...")
!pip install --quiet transformers accelerate bitsandbytes compressed-tensors sentencepiece pytest pytest-asyncio requests gitpython kaggle tabulate

import torch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    device_name = torch.cuda.get_device_name(0)
    print(f"[Device] ✅ GPU 検出: {device_name} (総 VRAM: {vram:.2f} GB)")
else:
    print("[Device ⚠️ WARNING] GPU が無効です。Colabメニューで『A100 GPU (ハイメモリ)』を選択してください。")

# Kaggle API による Day 16 (Ref 58038278) 提出ステータス確認
try:
    from google.colab import userdata
    k_user = userdata.get('KAGGLE_USERNAME')
    k_key = userdata.get('KAGGLE_KEY')
    if k_user and k_key:
        os.environ['KAGGLE_USERNAME'] = k_user
        os.environ['KAGGLE_KEY'] = k_key
        res = subprocess.run(['kaggle', 'competitions', 'submissions', '-c', 'gemma-4-developer-agent'], capture_output=True, text=True)
        print("\n[Kaggle Submission Status (Day 16)]:")
        for line in res.stdout.split('\n')[:5]:
            print("  " + line)
except Exception as e:
    print(f"[Kaggle Notice] Kaggle API チェック スキップ: {e}")


In [ ]:
# 2. サンドボックス環境の準備 (FastAPI リポジトリ & テスト実行環境)
SANDBOX_DIR = "/content/mini_kaggle_sandbox"
REPO_DIR = os.path.join(SANDBOX_DIR, "fastapi_repo")
os.makedirs(SANDBOX_DIR, exist_ok=True)

if not os.path.exists(REPO_DIR):
    print("[Sandbox] FastAPI 公式リポジトリをクローン中...")
    !git clone --quiet https://github.com/fastapi/fastapi.git {REPO_DIR}
else:
    print("[Sandbox] 既存の FastAPI リポジトリを使用します。")

print("[Sandbox] テスト実行用の依存関係をインストール中...")
!pip install --quiet pydantic starlette httpx uvicorn email_validator
print("[Sandbox] ✅ サンドボックス準備完了。")


In [ ]:
# 3. 本番同等モデルのロード (自己修復 & 自動フォールバック対応)
import subprocess
import sys

# compressed-tensors の自動インポート & 未インストールの場合は即時導入
try:
    import compressed_tensors
except ImportError:
    print("[Setup] compressed-tensors を即時インストール中...")
    !pip install --quiet "compressed-tensors>=0.15.0"
    import compressed_tensors

from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

hf_token = None
try:
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        os.environ['HF_TOKEN'] = hf_token
except Exception:
    pass

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
model = None
tokenizer = None
MODEL_ID = ""

# 優先度 1: A100-80GB 環境での 31B-QAT 公式モデル
if vram_gb >= 70.0:
    PRIMARY_ID = "google/gemma-4-31B-it-qat-w4a16-ct"
    print(f"[Model] 🚀 A100-80GB 検出 ({vram_gb:.1f} GB): 本番公式 31B モデル ({PRIMARY_ID}) をロード試行中...")
    try:
        tokenizer = AutoTokenizer.from_pretrained(PRIMARY_ID, token=hf_token)
        model = AutoModelForCausalLM.from_pretrained(
            PRIMARY_ID,
            device_map={0: "38GiB", "cpu": "60GiB"},
            torch_dtype=torch.bfloat16,
            token=hf_token
        )
        MODEL_ID = PRIMARY_ID
        print(f"[Model] 🎉 31B-QAT ロード成功！")
    except Exception as e:
        print(f"[Model Warning] 31B ロード失敗 ({e})。安全のため 27B-it 4-bit へ自動フォールバックします。")
        model = None

# 優先度 2: 27B-it 4-bit (A100-40GB または 31B 失敗時)
if model is None and vram_gb >= 35.0:
    FALLBACK_27B = "google/gemma-2-27b-it"
    print(f"[Model] 🚀 27B モデル ({FALLBACK_27B}) を 4-bit 直結ロード中...")
    try:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True
        )
        tokenizer = AutoTokenizer.from_pretrained(FALLBACK_27B, token=hf_token)
        model = AutoModelForCausalLM.from_pretrained(
            FALLBACK_27B,
            quantization_config=bnb_config,
            device_map={"": 0},
            torch_dtype=torch.bfloat16,
            token=hf_token
        )
        MODEL_ID = FALLBACK_27B
        print(f"[Model] 🎉 27B-it ロード成功！")
    except Exception as e2:
        print(f"[Model Warning] 27B ロード失敗 ({e2})。9B へ自動フォールバックします。")
        model = None

# 優先度 3: 9B-it 4-bit (T4 または非常時)
if model is None:
    FALLBACK_9B = "google/gemma-2-9b-it"
    print(f"[Model] 🚀 9B モデル ({FALLBACK_9B}) を 4-bit 直結ロード中...")
    bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16)
    tokenizer = AutoTokenizer.from_pretrained(FALLBACK_9B, token=hf_token)
    model = AutoModelForCausalLM.from_pretrained(FALLBACK_9B, quantization_config=bnb_config, device_map={"": 0}, torch_dtype=torch.bfloat16, token=hf_token)
    MODEL_ID = FALLBACK_9B
    print(f"[Model] ✅ 9B-it ロード完了。")

alloc = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0
print(f"[Model Ready] 🎯 確定稼働モデル: {MODEL_ID} | VRAM 消費: {alloc:.2f} GB / 空き: {vram_gb - alloc:.1f} GB")

def call_llm(messages, temperature=0.2, max_new_tokens=1024):
    formatted = []
    sys_prefix = ""
    for m in messages:
        if m["role"] == "system":
            sys_prefix += m["content"].strip() + "\n\n"
        elif m["role"] == "user":
            formatted.append({"role": "user", "content": sys_prefix + m["content"]})
            sys_prefix = ""
        else:
            formatted.append(m)
    if not formatted and sys_prefix:
        formatted.append({"role": "user", "content": sys_prefix})
        
    inputs = tokenizer.apply_chat_template(formatted, return_tensors="pt", tokenize=True).to(model.device)
    
    t0 = time.time()
    with torch.inference_mode():
        outputs = model.generate(
            inputs,
            max_new_tokens=max_new_tokens,
            do_sample=(temperature > 0.0),
            temperature=max(temperature, 1e-4),
            pad_token_id=tokenizer.eos_token_id
        )
    elapsed = time.time() - t0
    resp = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    return resp, elapsed

print("[Engine] ✅ 推論関数 call_llm 準備完了。")


In [ ]:
# 4. 実環境ツール実行エンジン (Real Tool Execution Engine)

def execute_tool(tool_name, arguments, repo_dir):
    """リポジトリサンドボックス上で実際のツールを実行する"""
    try:
        if tool_name == "read_file":
            p = os.path.join(repo_dir, arguments.get("path", ""))
            if not os.path.exists(p):
                return f"Error: File not found: {arguments.get('path')}"
            with open(p, "r", encoding="utf-8", errors="ignore") as f:
                lines = f.readlines()
            s = max(1, int(arguments.get("start_line", 1))) - 1
            e = min(len(lines), int(arguments.get("end_line", len(lines))))
            out = [f"{i+1}: {lines[i]}" for i in range(s, e)]
            return "".join(out)[:3000]
            
        elif tool_name == "edit_file":
            p = os.path.join(repo_dir, arguments.get("path", ""))
            if not os.path.exists(p):
                return f"Error: File not found: {arguments.get('path')}"
            old_str = arguments.get("old_string", "")
            new_str = arguments.get("new_string", "")
            if not old_str:
                return "Error: `old_string` is required and cannot be empty."
            with open(p, "r", encoding="utf-8") as f:
                content = f.read()
            if old_str not in content:
                return f"Error: `old_string` not found in {arguments.get('path')}. Make sure exact indentation and whitespace match."
            content = content.replace(old_str, new_str, 1)
            with open(p, "w", encoding="utf-8") as f:
                f.write(content)
            return f"Successfully edited {arguments.get('path')}."
            
        elif tool_name == "run_command":
            cmd = arguments.get("command", "")
            if not cmd:
                return "Error: command is required."
            res = subprocess.run(cmd, shell=True, cwd=repo_dir, capture_output=True, text=True, timeout=30)
            out = (res.stdout + "\n" + res.stderr).strip()
            return out[:2500] if out else "(No output, exit code: " + str(res.returncode) + ")"
            
        elif tool_name == "get_status":
            res = subprocess.run(["git", "status", "--short"], cwd=repo_dir, capture_output=True, text=True)
            return res.stdout.strip() if res.stdout.strip() else "No changes."
            
        elif tool_name == "submit_patch":
            return "SUBMIT_PATCH_CALLED"
            
        else:
            return f"Error: Unknown tool '{tool_name}'."
    except Exception as e:
        return f"Execution Error in {tool_name}: {e}"

def parse_tool_call(response_text):
    """モデルのテキストから <tool_call> を抽出・パースし、エラーフラグを検知する"""
    m = re.search(r'<tool_call>\s*(\{.*?\})\s*</tool_call>', response_text, re.DOTALL)
    if not m:
        # マークダウン json ブロックのフォールバック
        m = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', response_text, re.DOTALL)
    
    if not m:
        return None, {"has_tool": False, "error": "No tool tag found"}
        
    raw_json = m.group(1).strip()
    
    # 配列引数フリーズバグ (Topic 753973) の検知
    array_arg_detected = bool(re.search(r'"arguments"\s*:\s*\[', raw_json))
    
    try:
        data = json.loads(raw_json)
        name = data.get("name", "")
        args = data.get("arguments", {})
        
        # old_string 欠損バグ (Topic 756051) の検知
        missing_old_string = (name == "edit_file" and "old_string" not in args)
        
        return {"name": name, "arguments": args}, {
            "has_tool": True,
            "array_arg": array_arg_detected,
            "missing_old_string": missing_old_string,
            "parse_ok": True
        }
    except json.JSONDecodeError as e:
        return None, {
            "has_tool": True,
            "parse_ok": False,
            "error": f"JSONDecodeError: {e}",
            "array_arg": array_arg_detected
        }

print("[Engine] ✅ 実環境ツールエンジン & 厳密パーサー準備完了。")


In [ ]:
# 5. A/B テスト用プロンプト・バリアントの定義 (A vs B vs C)

# --- [Variant 0: Baseline / 現行 V9.2 本番プロンプト] ---
PROMPT_VARIANT_0 = """You are an expert autonomous software engineering agent powered by Gemma 4.
Your objective is to resolve issues in code repositories by localizing bugs, developing minimal and robust fixes, verifying them against test suites, and submitting clean patches.

# Operational Principles
1. Explore Before Modifying: Inspect the issue and identify reproducing conditions. Use search before reading entire files.
2. Minimal & Targeted Changes: Do NOT refactor unrelated code. Preserve existing architecture and interfaces.
3. Rigorous Verification Loop: Locate tests, run with run_command to verify failure, apply patch with edit_file, re-run test.
4. Completion: Call submit_patch to finish the task.

Available tools: read_file, edit_file, run_command, get_status, submit_patch.
Format: <tool_call>{"name": "...", "arguments": {...}}</tool_call>
"""

# --- [Variant 1: B 案 / 最小差分安全版 (One-Shot JSON + 配列引数禁止)] ---
PROMPT_VARIANT_1 = PROMPT_VARIANT_0 + """
CRITICAL TOOL CALL FORMAT RULES:
1. NEVER pass lists or arrays as tool arguments. Pass a single flat string.
2. You MUST include BOTH `old_string` and `new_string` in EVERY `edit_file` call.

EXACT TOOL EXAMPLE:
<tool_call>
{"name": "edit_file", "arguments": {"path": "fastapi/routing.py", "old_string": "def run(self):", "new_string": "def run(self, safe: bool = True):"}}
</tool_call>
"""

# --- [Variant 2: C 案検証版 (Variant 1 + Concise Reasoning 2文制限)] ---
PROMPT_VARIANT_2 = PROMPT_VARIANT_1 + """
3. Concise reasoning: Write at most 2 sentences of explanation before emitting tool calls.
"""

VARIANTS = {
    "V0_Baseline_V9.2": PROMPT_VARIANT_0,
    "V1_MinDiff_B_OneShot": PROMPT_VARIANT_1,
    "V2_Concise_C_2Sentences": PROMPT_VARIANT_2
}
print("[Prompt] ✅ 3 大バリアント定義完了:", list(VARIANTS.keys()))


In [ ]:
# 6. LocBench 実タスクデータセットの取得 & タスク選定
import urllib.request

LOCBENCH_FILE = "/content/gemma4-locbench-v1.json"
LOCBENCH_URL = "https://raw.githubusercontent.com/chiba244qq/gemma4-agent-dashboard/main/data/locbench/gemma4-locbench-v1.json"

if not os.path.exists(LOCBENCH_FILE):
    print("[LocBench] GitHub から LocBench-v1 をダウンロード中...")
    urllib.request.urlretrieve(LOCBENCH_URL, LOCBENCH_FILE)

with open(LOCBENCH_FILE, "r", encoding="utf-8") as f:
    locbench_data = json.load(f)

all_records = locbench_data.get("records", [])
fastapi_records = [r for r in all_records if r.get("repo") == "fastapi/fastapi"]
print(f"[LocBench] ✅ 読み込み完了: 全 {len(all_records)} 件中、FastAPI タスク: {len(fastapi_records)} 件")

# 実行モード設定: 'smoke' (代表 3 タスク / 約 5-10 分) または 'benchmark' (代表 10 タスク / 約 30-40 分)
EVAL_MODE = "smoke"   # << 必要に応じて 'benchmark' に変更してください

if EVAL_MODE == "smoke":
    SELECTED_TASKS = fastapi_records[:3]  # [fastapi_15661, fastapi_15588, fastapi_15589]
else:
    SELECTED_TASKS = fastapi_records[:10]

print(f"[LocBench] 🎯 選択された評価タスク件数 ({EVAL_MODE} モード): {len(SELECTED_TASKS)} 件")
for t in SELECTED_TASKS:
    print(f"  - {t['instance_id']}: {t.get('verification_test_files')} ({t.get('problem_statement_excerpt')[:60]}...)")


In [ ]:
# 7. 【真の A/B 検証実行エンジン】マルチターン自律対話 & pytest 実機判定ループ

def evaluate_task_variant(task, variant_name, system_prompt, max_turns=8):
    """1 タスク × 1 バリアントの完全な自律エージェントループを実行し、pytest で合否を判定する"""
    print(f"\n{'='*65}")
    print(f"▶ Task: {task['instance_id']} | Variant: {variant_name}")
    print(f"{'='*65}")
    
    # 1. リポジトリを完全クリーンアップ & 当該コミットにリセット
    subprocess.run(["git", "checkout", "-f", task["base_commit"]], cwd=REPO_DIR, capture_output=True)
    subprocess.run(["git", "clean", "-fd"], cwd=REPO_DIR, capture_output=True)
    
    # 初期プロンプト作成
    issue_text = f"Issue Description for {task['instance_id']}:\n" + task.get("problem_statement_excerpt", "Fix the bug in the codebase.")
    if task.get("verification_test_files"):
        issue_text += f"\nRelated test files to verify: {', '.join(task['verification_test_files'])}"
        
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": issue_text}
    ]
    
    turn_logs = []
    total_latency = 0.0
    parse_errors = 0
    array_arg_count = 0
    missing_old_str_count = 0
    patch_applied = False
    
    for turn in range(1, max_turns + 1):
        # Call 8 での強制編集トリガー (V9.2 ロジックの再現)
        if turn == 8:
            messages.append({"role": "user", "content": "[SYSTEM REMINDER: Call 8/8 reached. You MUST call edit_file now to apply your minimal bugfix.]"})
            
        resp, lat = call_llm(messages, temperature=0.2, max_new_tokens=1024)
        total_latency += lat
        print(f"  [Turn {turn}/{max_turns}] 推論所要: {lat:.2f}s | 出力長: {len(resp)} 文字")
        
        tool_call, audit = parse_tool_call(resp)
        if audit.get("array_arg"):
            array_arg_count += 1
        if audit.get("missing_old_string"):
            missing_old_str_count += 1
        if not audit.get("parse_ok", True):
            parse_errors += 1
            
        if not tool_call:
            print(f"    ⚠️ ツール呼出なし: {resp[:120]}...")
            messages.append({"role": "model", "content": resp})
            messages.append({"role": "user", "content": "Please emit a valid <tool_call>{"name": "...", "arguments": {...}}</tool_call>."})
            continue
            
        t_name = tool_call.get("name", "")
        t_args = tool_call.get("arguments", {})
        print(f"    🛠️ Tool Call: {t_name}({list(t_args.keys())})")
        
        if t_name == "submit_patch":
            print("    🏁 submit_patch が呼び出されました。探索を終了します。")
            break
            
        # ツールを実際に実行
        tool_out = execute_tool(t_name, t_args, REPO_DIR)
        if t_name == "edit_file" and "Successfully edited" in tool_out:
            patch_applied = True
            
        messages.append({"role": "model", "content": resp})
        messages.append({"role": "user", "content": f"Tool output:\n{tool_out}"})
    
    # 2. 実際の git diff を取得
    diff_res = subprocess.run(["git", "diff"], cwd=REPO_DIR, capture_output=True, text=True)
    diff_text = diff_res.stdout.strip()
    has_diff = len(diff_text) > 0
    
    # 3. 実際の pytest を実行して合否を厳密判定
    test_files = task.get("verification_test_files", [])
    pytest_pass = False
    pytest_log = ""
    
    if test_files and os.path.exists(os.path.join(REPO_DIR, test_files[0])):
        cmd = ["pytest", "-q", "--maxfail=1"] + test_files
        py_res = subprocess.run(cmd, cwd=REPO_DIR, capture_output=True, text=True, timeout=60)
        pytest_pass = (py_res.returncode == 0)
        pytest_log = (py_res.stdout + "\n" + py_res.stderr)[-600:].strip()
    else:
        pytest_log = "No test files found."
        
    print(f"  📊 Task 結果: パッチ生成={has_diff} | pytest判定={'✅ PASS' if pytest_pass else '❌ FAIL'}")
    
    return {
        "task_id": task["instance_id"],
        "variant": variant_name,
        "pytest_pass": pytest_pass,
        "has_diff": has_diff,
        "diff_lines": len(diff_text.split('\n')) if has_diff else 0,
        "turns_used": turn,
        "total_latency_sec": round(total_latency, 2),
        "avg_step_latency_sec": round(total_latency / max(1, turn), 2),
        "parse_errors": parse_errors,
        "array_arg_count": array_arg_count,
        "missing_old_str_count": missing_old_str_count,
        "pytest_log": pytest_log
    }

# 全タスク × 全バリアントの A/B 実行
all_eval_results = []
for t in SELECTED_TASKS:
    for v_name, v_prompt in VARIANTS.items():
        res = evaluate_task_variant(t, v_name, v_prompt, max_turns=8)
        all_eval_results.append(res)
print("\n🎉 全 A/B 実機検証ランが完了しました！")


In [ ]:
# 8. 結果集計・統計比較表の出力 & GitHub 自動同期
import pandas as pd
from tabulate import tabulate

df = pd.DataFrame(all_eval_results)

# バリアントごとの集計
summary = []
for v_name in VARIANTS.keys():
    v_df = df[df["variant"] == v_name]
    n = len(v_df)
    if n == 0:
        continue
    pass_rate = (v_df["pytest_pass"].sum() / n) * 100
    patch_rate = (v_df["has_diff"].sum() / n) * 100
    avg_latency = v_df["avg_step_latency_sec"].mean()
    avg_turns = v_df["turns_used"].mean()
    total_parse_err = v_df["parse_errors"].sum()
    total_array_err = v_df["array_arg_count"].sum()
    total_missing_old = v_df["missing_old_str_count"].sum()
    
    summary.append({
        "Variant": v_name,
        "Tasks": n,
        "Pass Rate (%)": f"{pass_rate:.1f}%",
        "Patch Rate (%)": f"{patch_rate:.1f}%",
        "Parse Errors": int(total_parse_err),
        "Array Arg Hits": int(total_array_err),
        "Missing old_string": int(total_missing_old),
        "Avg Step Latency (s)": f"{avg_latency:.2f}s",
        "Avg Turns": f"{avg_turns:.1f}"
    })

sum_df = pd.DataFrame(summary)
print("="*85)
print("🏆 【真の A/B 検証結果サマリー】(実パッチ pytest 判定 & マルチターン自律実行)")
print("="*85)
print(tabulate(sum_df, headers="keys", tablefmt="github", showindex=False))

# GitHub への自動同期
now_str = time.strftime("%Y-%m-%d %H:%M:%S JST")
output_data = {
    "timestamp": now_str,
    "model": MODEL_ID,
    "eval_mode": EVAL_MODE,
    "summary": summary,
    "detailed_results": all_eval_results
}

try:
    github_pat = userdata.get('GITHUB_PAT')
    if github_pat:
        import requests, base64
        repo = "chiba244qq/gemma4-agent-dashboard"
        path = "data/locbench_ab_test_results.json"
        api_url = f"https://api.github.com/repos/{repo}/contents/{path}"
        headers = {"Authorization": f"token {github_pat}", "Accept": "application/vnd.github.v3+json"}
        
        sha = None
        get_res = requests.get(api_url, headers=headers)
        if get_res.status_code == 200:
            sha = get_res.json().get("sha")
            
        content_bytes = json.dumps(output_data, indent=2, ensure_ascii=False).encode('utf-8')
        payload = {
            "message": f"[LocBench A/B Sync] 真の実パッチ pytest 評価結果更新 ({now_str})",
            "content": base64.b64encode(content_bytes).decode('utf-8')
        }
        if sha:
            payload["sha"] = sha
            
        put_res = requests.put(api_url, headers=headers, json=payload)
        if put_res.status_code in [200, 201]:
            print(f"\n[Sync] 🎉 GitHub ({path}) への自動プッシュに成功しました！")
        else:
            print(f"\n[Sync Warning] GitHub プッシュ失敗: {put_res.status_code} - {put_res.text}")
except Exception as e:
    print(f"\n[Sync Notice] GitHub 同期スキップ: {e}")
